In [10]:
import numpy as np
w_hh=np.full((3,2),1)
w_hx=np.full((3,3),9)
print(w_hh)
print(w_hh.shape)
print(w_hx)
print(w_hx.shape)

[[1 1]
 [1 1]
 [1 1]]
(3, 2)
[[9 9 9]
 [9 9 9]
 [9 9 9]]
(3, 3)


In [11]:
w_h1=np.concatenate((w_hh,w_hx),axis=1)
print(w_h1)
print(w_h1.shape)

[[1 1 9 9 9]
 [1 1 9 9 9]
 [1 1 9 9 9]]
(3, 5)


In [12]:
w_h2=np.hstack((w_hh,w_hx))
print(w_h2)
print(w_h2.shape)

[[1 1 9 9 9]
 [1 1 9 9 9]
 [1 1 9 9 9]]
(3, 5)


In [13]:
h_t_prev=np.full((2,1),1)
x_t=np.full((3,1),9)

print(h_t_prev)
print(h_t_prev.shape)
print(x_t)
print(x_t.shape)

[[1]
 [1]]
(2, 1)
[[9]
 [9]
 [9]]
(3, 1)


In [14]:
ax_1=np.concatenate((h_t_prev,x_t),axis=0)
print(ax_1)
print(ax_1.shape)

[[1]
 [1]
 [9]
 [9]
 [9]]
(5, 1)


In [15]:
ax_2=np.vstack((h_t_prev,x_t))
print(ax_2)
print(ax_2.shape)

[[1]
 [1]
 [9]
 [9]
 [9]]
(5, 1)


In [16]:
w_hh=np.full((3,2),1)
w_hx=np.full((3,3),9)
h_t_prev=np.full((2,1),1)
x_t=np.full((3,1),9)

In [17]:
stack_1=np.hstack((w_hh,w_hx))
stack_2=np.vstack((h_t_prev,x_t))

print(stack_1)
print(stack_2)
formula_1=np.matmul(np.hstack((w_hh,w_hx)),np.vstack((h_t_prev,x_t)))
print(formula_1)

[[1 1 9 9 9]
 [1 1 9 9 9]
 [1 1 9 9 9]]
[[1]
 [1]
 [9]
 [9]
 [9]]
[[245]
 [245]
 [245]]


In [18]:
mul_1=np.matmul(w_hh,h_t_prev)
mul_2=np.matmul(w_hx,x_t)
print(mul_1)
print(mul_2)
formula_2=np.matmul(w_hh,h_t_prev)+np.matmul(w_hx,x_t)
print(formula_2)
print(np.allclose(formula_1,formula_2))

[[2]
 [2]
 [2]]
[[243]
 [243]
 [243]]
[[245]
 [245]
 [245]]
True


In [19]:
import numpy as np
from numpy import random
from time import perf_counter
import tensorflow as tf

In [20]:
def sigmoid(x):
    return 1.0/(1.0+np.exp(-x))

In [21]:
random.seed=(10)
emb=128
T=256
h_dim=16
h_0=np.zeros((h_dim,1))

w1=random.standard_normal((h_dim,emb+h_dim))
w2=random.standard_normal((h_dim,emb+h_dim))
w3=random.standard_normal((h_dim,emb+h_dim))
b1=random.standard_normal((h_dim,1))
b2=random.standard_normal((h_dim,1))
b3=random.standard_normal((h_dim,1))

x=random.standard_normal((T,emb,1))

weights_vanilla=[w1,b1]
weights_GRU=[w1.copy(),w2,w3,b1.copy(),b2,b3]

print(x)

[[[-0.93336025]
  [ 0.3018903 ]
  [ 0.59581759]
  ...
  [-1.57436921]
  [-1.01767272]
  [ 0.83986087]]

 [[-0.80150821]
  [ 1.26290086]
  [-0.83451475]
  ...
  [-0.18900249]
  [ 0.1413946 ]
  [ 1.32096973]]

 [[ 0.79999968]
  [-0.23965733]
  [-0.82458901]
  ...
  [-0.45188504]
  [ 0.03497897]
  [ 0.13567294]]

 ...

 [[-0.4287035 ]
  [ 0.12371137]
  [-1.26343216]
  ...
  [ 1.38568049]
  [-1.34119717]
  [-0.83475378]]

 [[ 0.26046065]
  [-0.25052842]
  [ 1.17424022]
  ...
  [ 1.34846141]
  [-0.97587778]
  [-0.24123895]]

 [[ 0.2583535 ]
  [ 0.79541423]
  [-0.22121038]
  ...
  [ 0.79460199]
  [ 1.89029555]
  [-0.90869235]]]


In [22]:
def forward_V_RNN(inputs, weights):
    x, h_t = inputs
    wh, bh = weights
    h_t = np.dot(wh, np.concatenate([h_t, x])) + bh
    h_t = sigmoid(h_t)
    y = h_t
    return y, h_t

In [23]:
def forward_GRU(inputs, weights):
    x, h_t = inputs
    wu, wr, wc, bu, br, bc = weights
    u = np.dot(wu, np.concatenate([h_t, x])) + bu
    u = sigmoid(u)
    r = np.dot(wr, np.concatenate([h_t, x])) + br
    r = sigmoid(r)
    c = np.dot(wc, np.concatenate([r*h_t, x])) + bc
    c = np.tanh(c)
    h_t = u*c + (1-u)*h_t
    y = h_t
    return y, h_t

In [24]:
forward_GRU([x[1], h_0], weights_GRU)[0]

array([[ 2.51443356e-05],
       [ 9.99999999e-01],
       [ 7.62168597e-04],
       [ 5.98924500e-01],
       [-3.71524860e-02],
       [-2.84490272e-05],
       [ 6.13095110e-08],
       [-9.99999997e-01],
       [ 1.78874089e-03],
       [-9.32067658e-02],
       [ 5.04906072e-05],
       [-1.71879919e-03],
       [ 2.29343193e-02],
       [-9.90510219e-01],
       [-9.99955431e-01],
       [-6.55155514e-01]])

In [25]:
def scan(fn, elems, weights, h_0):
    h_t = h_0
    ys = []
    for x in elems:
        y, h_t = fn([x, h_t], weights)
        ys.append(y)
    return ys, h_t

In [26]:
ys, h_t = scan(forward_V_RNN, x, 
weights_vanilla, h_0)
print(f"Length of ys:{len(ys)}")
print(f"Shape of each y within ys:{ys[0].shape}")
print(f"Shape of h_t:{h_t.shape}")

Length of ys:256
Shape of each y within ys:(16, 1)
Shape of h_t:(16, 1)


In [27]:
tic = perf_counter()
ys , h_t = scan(forward_V_RNN,x, weights_vanilla, h_0)
toc = perf_counter()
RNN_time = (toc-tic)*1000
print(f"It tok {RNN_time :.2f}ms to run the forward method for the vanilla RNN.")

tic = perf_counter()
ys , h_t = scan(forward_GRU,x, weights_GRU, h_0)
toc = perf_counter()
GRU_time = (toc-tic)*1000
print(f"It tok {GRU_time :.2f}ms to run the forward method for the vanilla RNN.")

It tok 5.96ms to run the forward method for the vanilla RNN.
It tok 18.36ms to run the forward method for the vanilla RNN.


In [28]:
model_GRU = tf.keras.Sequential([
    tf.keras.layers.GRU(256, return_sequences=True, name='GRU_1_returns_seq'),
    tf.keras.layers.GRU(128, return_sequences=True, name='GRU_2_returns_seq'),
    tf.keras.layers.GRU(64, name='GRU_3_returns_last_only'),
    tf.keras.layers.Dense(10)  
])


In [29]:
try:
    model_GRU.summary()
except Exception as e:
    print(e) 


This model has not yet been built. Build the model first by calling `build()` or by calling the model on a batch of data.


In [30]:
batch_size=60
sequence_lenght=50
word_vector_lenght=40

input_data=tf.random.normal([batch_size,sequence_lenght,word_vector_lenght])
prediction=model_GRU(input_data)
model_GRU.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 GRU_1_returns_seq (GRU)     (60, 50, 256)             228864    
                                                                 
 GRU_2_returns_seq (GRU)     (60, 50, 128)             148224    
                                                                 
 GRU_3_returns_last_only (GR  (60, 64)                 37248     
 U)                                                              
                                                                 
 dense (Dense)               (60, 10)                  650       
                                                                 
Total params: 414,986
Trainable params: 414,986
Non-trainable params: 0
_________________________________________________________________


In [31]:
new_word_vector_lenght=44
input_data_1=tf.random.normal([batch_size,sequence_lenght,new_word_vector_lenght])

try:
    prediction=model_GRU(input_data_1)
except Exception as e:
    print(e) 

Exception encountered when calling layer 'sequential' (type Sequential).

Input 0 of layer "GRU_1_returns_seq" is incompatible with the layer: expected shape=(None, None, 40), found shape=(60, 50, 44)

Call arguments received by layer 'sequential' (type Sequential):
  • inputs=tf.Tensor(shape=(60, 50, 44), dtype=float32)
  • training=None
  • mask=None


In [ ]:
import os

import trax
import trax.fastmath.numpy as np

import pickle
import numpy
import random as rnd

from trax import fastmath
from trax import layers as tl

trax.supervised.trainer_lib.init_random_number_generators(32)
random.seed(32)

In [ ]:
# dirname='data/'
# filename=''
# lines=[]
# counter=0

# with open(os.path.join(dirname,filename)) as files:
#     for line in files:
#         pure_line=line.strip()
        
#         if pure_line:
#             lines.append(pure_line)
# n_lines=len(lines)
# print(n_lines)

In [ ]:
# Specify the full path to the file directly
filename = 'shakespeare_data.txt'  # Replace 'your_file.txt' with the actual filename
lines = []
counter = 0

try:
    with open(filename) as files:
        for line in files:
            pure_line = line.strip()
            if pure_line:
                lines.append(pure_line)

    n_lines = len(lines)
    print(n_lines)
except FileNotFoundError:
    print(f"The file {filename} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
for i ,line in enumerate(lines):
    lines[i]=line.lower()
    
print(n_lines)
print(lines[0])
print(lines[999])


In [ ]:
eval_lines=lines[-1000:]
print(len(lines))
print(len(eval_lines))


In [ ]:
def line_to_tensor(line,EOS_int=1):
    tensor=[]
    
    for c in line:
        c_int=ord(c)
        tensor.append(c_int)
    tensor.append(EOS_int)
    return tensor 

In [ ]:
line_to_tensor('abc xyz')

In [ ]:
# Specify the full path to the file directly
filename = 'shakespeare_data.txt'  # Replace with the actual filename
lines = []
counter = 0

try:
    with open(filename) as files:
        for line in files:
            pure_line = line.strip()
            if pure_line:
                lines.append(pure_line)

    n_lines = len(lines)
    print(n_lines)
except FileNotFoundError:
    print(f"The file {filename} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

# Convert lines to lowercase
for i, line in enumerate(lines):  # Fixed syntax error: 'for i line in' should be 'for i, line in'
    lines[i] = line.lower()

print(n_lines)
print(lines[0])  # Print the first line
print(lines[999])  # Print the 1000th line, if it exists

# Extract the last 1000 lines for evaluation
eval_lines = lines[-1000:]
print(len(lines))      # Total number of lines
print(len(eval_lines)) # Number of evaluation lines

# Define the EOS token (End Of Sentence) value
EOS_int = 1  # You can change this value if needed

def line_to_tensor(line, EOS_int=1):  # Added the parameter 'line' to the function
    tensor = []
    
    for c in line:
        c_int = ord(c)  # Get ASCII value of the character
        tensor.append(c_int)
    tensor.append(EOS_int)  # Append the EOS token
    return tensor 

# Test the line_to_tensor function with a sample line
tensor_result = line_to_tensor('abc xyz')
print(tensor_result)  # Print the tensor result


In [ ]:
import numpy as np
import random as rnd

def data_generator(batch_size, max_lenght, data_lines, line_to_tensor=lambda x: list(map(int, x)), shuffle=True):
    index = 0
    cur_batch = []
    num_lines = len(data_lines)
    
    lines_index = [*range(num_lines)]
    if shuffle:
        rnd.shuffle(lines_index)
    
    while True:
        if index >= num_lines:
            index = 0
            if shuffle:
                rnd.shuffle(lines_index)
        
        line = data_lines[lines_index[index]]
        if len(line) < max_lenght:
            cur_batch.append(line)
        
        index += 1
        
        if len(cur_batch) == batch_size:
            batch = []
            mask = []
            
            for li in cur_batch:
                tensor = line_to_tensor(li)
                pad = [0] * (max_lenght - len(tensor))
                tensor_pad = tensor + pad
                
                batch.append(tensor_pad)
                
                example_mask = [0 if t == 0 else 1 for t in tensor_pad]
                mask.append(example_mask)
            
            batch_np_arr = np.array(batch)
            mask_np_arr = np.array(mask)
            yield batch_np_arr, batch_np_arr, mask_np_arr
            
            cur_batch = []

In [ ]:
# Example usage
tmp_lines = ['12345678901', '123456789', '234567890', '345678901']
tmp_data_gen = data_generator(batch_size=2, max_lenght=10, data_lines=tmp_lines, shuffle=False)

tmp_batch = next(tmp_data_gen)
tmp_batch

In [ ]:
import itertools

# Define the infinite generator
infinite_data_generator = itertools.cycle(
    data_generator(batch_size=2, max_lenght=10, data_lines=tmp_lines)
)


In [ ]:
ten_lines=[next(infinite_data_generator) for _ in range(10)]
print(len(ten_lines))

In [ ]:
def GRULM(vocab_size=256,d_model=512,n_layers=2,mode='train'):
    model=t1.Serial(
    t1.ShiftRight(mode=mode),
    t1.Embedding(vocab_size=vocab_size,d_feature=d_model),
    [t1.GRU(n_units=d_mode) for _ in range(n_layers)],
    t1.Dense(n_units=vocab_size),
    t1.LogSoftmax()
    )
    
    return model

In [ ]:
model=GRULM()
print(model)

In [ ]:
from trax.supervised import training 

In [ ]:
def train_model(model,data_generator,)

In [ ]:
bare_train_generator=data_generator(batch_size,max_lenght,data_lines=lines)
infinite_train_generator=itertools.cycle(bare_train_generator)

bare_eval_generator=data_generator(batch_size,max_lenght,data_lines=eval_lines)
infinite_eval_generator=itertools.cycle(bare_eval_generator)

train_task=training.TrainTask(
labeled_data=infinite_train_genertator,
    loss_layer=t1.CrossEntropyLoss(),
    optimizer=trax.optimizers.Adam(0.0005)
)

eval_task=training.EvalTask(
labeled_data=[t1.CrossEntropyLoss(),t1.Accuracy()],
    n_eval_batches=3)

training_loop=training.Loop(model,train_task,eval_task=eval_task,output_dir=output_dir)

training_loop.run(n_steps=n_steps)

return training_loop


In [ ]:
trainin_loop=train_model(GRULM(),data_generator)

In [ ]:
def test_model(preds,target):
    total_loop_ppx=np.sum(preds*t1.one_hot(target,preds.shape[-1]),axis=-1)
    
    non_pad=1.0*np.equal(target,0)
    ppx=total_log_ppx*non_pad
    
    log_ppx=np.sum(ppx)/np.sum(non_pad)
    
    return -log_ppx

In [ ]:
model=GRULM()
model.init_fronm_file('model.pkl.gz')
batch=next(data_generator(batch_size,max_lenght,lines,shuffle=False))
preds=mode(batch[0])
log_ppx=test_mode(preds,batch[1])
print(log_ppx)

In [ ]:
import numpy as np

In [ ]:
def gumbel_sample(log_probs, temperature=1.0):
    u = np.random.uniform(low=1e-10, high=1.0-1e-10, size=log_probs.shape)
    g = -np.log(np.log(u))
    return np.argmax(log_probs + g* temperature, axis=-1)

def predict(num_chars, prefix):
    inp = [ord(C) for c in prefix]
    result= [c for c in prefix]
    max_len = len(prefix) + num_chars
    for _ in range(num_chars):
        cur_inp = np.array(inp + [0]*(max_len - len(inp)))
        outp = model(cur_inp[None,:])
        next_char = gumbel_sample(outp[0, len(inp)])
        inp += [int(next_char)]
        if inp[-1] == 1:
              break
        result.append(char(int(next_char)))
    return "".join(result)

print(predict(32, ""))